In [ ]:
# ============================================================
# 01_data_processing_colab
# Cloud-Based Intelligent Taekwondo Motion Classification System
#
# Purpose:
# 1. Scan original taekwondo videos from Google Drive
# 2. Build video manifest
# 3. Clean invalid / unwanted videos
# 4. Summarize extracted keypoint CSV files
# 5. Merge video manifest with keypoint CSV summary
# 6. Generate rule-based classification results
# 7. Create manual labeling template
# 8. Create manual labeling sample with original video paths
# ============================================================


# ============================================================
# 0. Safe Google Drive Mount
# ============================================================

import os

if os.path.exists("/content/drive/MyDrive"):
    print("Google Drive is already mounted.")
else:
    from google.colab import drive
    drive.mount("/content/drive")


# ============================================================
# 1. Imports and Global Paths
# ============================================================

import re
import glob
import numpy as np
import pandas as pd
from IPython.display import display


ROOT = "/content/drive/MyDrive"

# Original video folder
VIDEO_ROOT = f"{ROOT}/taekwondo_videos"

# Keypoint CSV folder generated by MediaPipe main actor extraction
KEYPOINT_CSV_DIR = f"{ROOT}/taekwondo_keypoints_csv_main_actor"

# Output files
MANIFEST_PATH = f"{ROOT}/taekwondo_manifest.csv"
CLEAN_MANIFEST_PATH = f"{ROOT}/taekwondo_manifest_clean.csv"

KEYPOINT_SUMMARY_PATH = f"{ROOT}/taekwondo_keypoints_csv_summary.csv"
MISSING_KEYPOINTS_PATH = f"{ROOT}/taekwondo_missing_keypoints.csv"
MERGED_MANIFEST_PATH = f"{ROOT}/taekwondo_keypoints_merged_manifest.csv"

RULE_BASED_RESULTS_PATH = f"{ROOT}/taekwondo_rule_based_classification_results.csv"

MANUAL_LABEL_TEMPLATE_PATH = f"{ROOT}/taekwondo_manual_labeling_template.csv"
MANUAL_LABEL_SAMPLE_PATH = f"{ROOT}/taekwondo_manual_labeling_sample_120_with_video_path.csv"


VIDEO_EXTS = (
    ".mp4", ".MP4",
    ".mov", ".MOV",
    ".avi", ".AVI",
    ".mkv", ".MKV"
)

SKIP_FILES = {
    "IMG_6178",
}

RANDOM_STATE = 42


print("VIDEO_ROOT:", VIDEO_ROOT)
print("KEYPOINT_CSV_DIR:", KEYPOINT_CSV_DIR)


# ============================================================
# 2. Optional: Print Video Folder Tree
# ============================================================

def print_tree(start_path, max_depth=3):
    """
    Print a simple folder tree for checking Google Drive structure.
    """
    if not os.path.exists(start_path):
        print("Folder not found:", start_path)
        return

    for root, dirs, files in os.walk(start_path):
        level = root.replace(start_path, "").count(os.sep)
        if level > max_depth:
            continue

        indent = " " * 4 * level
        print(f"{indent}📁 {os.path.basename(root)}")

        sub_indent = " " * 4 * (level + 1)
        for f in files[:20]:
            print(f"{sub_indent}📄 {f}")

        if len(files) > 20:
            print(f"{sub_indent}... ({len(files) - 20} more files)")


print("\nFolder preview:")
print_tree(VIDEO_ROOT, max_depth=2)


# ============================================================
# 3. Build Original Video Manifest
# ============================================================

def guess_action_label(file_name, path_text):
    """
    Guess action label from filename/path keywords.
    This is only metadata support, not final training label.
    """
    text = f"{file_name} {path_text}".lower()

    if "正踢" in text or "前踢" in text or "front" in text:
        return "front_kick"
    elif "旋踢" in text or "roundhouse" in text or "round" in text:
        return "roundhouse_kick"
    elif "側踢" in text or "側踩" in text or "侧踢" in text or "side" in text:
        return "side_kick"
    elif "下壓" in text or "下压" in text or "axe" in text:
        return "axe_kick"
    elif "後踢" in text or "后踢" in text or "back" in text:
        return "back_kick"
    elif "後旋" in text or "后旋" in text:
        return "spinning_hook_kick"
    elif "正拳" in text or "punch" in text:
        return "straight_punch"
    else:
        return "unknown"


def is_two_minute_video(file_name, path_text):
    """
    Detect long drill videos by filename/path keywords.
    """
    text = f"{file_name} {path_text}".lower()

    keywords = [
        "兩分鐘",
        "2分鐘",
        "2 minute",
        "2-minute",
        "two minute",
        "two-minute",
        "2min",
        "2_min",
    ]

    return any(k.lower() in text for k in keywords)


def extract_student_id(parts):
    """
    Extract student ID like U11313035 from path parts.
    """
    for p in parts:
        m = re.search(r"(U\d{8})", p)
        if m:
            return m.group(1)
    return None


def build_video_manifest(video_root):
    """
    Scan all video files under VIDEO_ROOT and build manifest dataframe.
    """
    rows = []

    if not os.path.exists(video_root):
        raise FileNotFoundError(f"Video root not found: {video_root}")

    for root, dirs, files in os.walk(video_root):
        for f in files:
            if not f.endswith(VIDEO_EXTS):
                continue

            full_path = os.path.join(root, f)
            rel_path = os.path.relpath(full_path, video_root)
            parts = rel_path.split(os.sep)

            top_level_folder = parts[0] if len(parts) > 0 else None
            second_level_folder = parts[1] if len(parts) > 1 else None
            third_level_folder = parts[2] if len(parts) > 2 else None
            fourth_level_folder = parts[3] if len(parts) > 3 else None

            student_id = extract_student_id(parts)
            ext = os.path.splitext(f)[1]

            rows.append({
                "file_name": f,
                "full_path": full_path,
                "relative_path": rel_path,
                "top_level_folder": top_level_folder,
                "second_level_folder": second_level_folder,
                "third_level_folder": third_level_folder,
                "fourth_level_folder": fourth_level_folder,
                "student_id": student_id,
                "file_ext": ext,
                "is_two_minute": is_two_minute_video(f, rel_path),
                "action_label_guess": guess_action_label(f, rel_path),
            })

    return pd.DataFrame(rows)


df_manifest = build_video_manifest(VIDEO_ROOT)

df_manifest.to_csv(MANIFEST_PATH, index=False, encoding="utf-8-sig")

print("\nOriginal video manifest saved:")
print(MANIFEST_PATH)
print("Total videos found:", len(df_manifest))

display(df_manifest.head(10))


# ============================================================
# 4. Clean Manifest
# ============================================================

def is_valid_video(row):
    """
    Remove unwanted or invalid video files.
    """
    fname = str(row["file_name"]).strip()

    # Remove two-minute drill videos
    if bool(row["is_two_minute"]):
        return False

    # Remove macOS resource fork files
    if fname.startswith("._"):
        return False

    # Remove abnormal empty-like names
    if fname in [".MOV", ".mov", "(1).MOV", "(1).mov"]:
        return False

    # Remove converted duplicate artifact files
    if "AnyConv.com__" in fname:
        return False

    return True


df_clean = df_manifest[df_manifest.apply(is_valid_video, axis=1)].copy()

df_clean.to_csv(CLEAN_MANIFEST_PATH, index=False, encoding="utf-8-sig")

print("\nClean manifest saved:")
print(CLEAN_MANIFEST_PATH)
print("Original videos:", len(df_manifest))
print("Clean videos:", len(df_clean))

print("\nClean manifest action label guess distribution:")
print(df_clean["action_label_guess"].value_counts(dropna=False))

display(df_clean.head(10))


# ============================================================
# 5. Summarize Generated Keypoint CSV Files
# ============================================================

def build_keypoint_csv_summary(keypoint_csv_dir):
    """
    Summarize generated keypoint CSV files.
    """
    rows = []

    if not os.path.exists(keypoint_csv_dir):
        print("Keypoint CSV folder not found:", keypoint_csv_dir)
        return pd.DataFrame(columns=["csv_file", "csv_path", "video_name", "size_mb"])

    for f in os.listdir(keypoint_csv_dir):
        if not f.endswith(".csv"):
            continue

        path = os.path.join(keypoint_csv_dir, f)

        rows.append({
            "csv_file": f,
            "csv_path": path,
            "video_name": f.replace("_keypoints.csv", ""),
            "size_mb": os.path.getsize(path) / 1024 / 1024,
        })

    return pd.DataFrame(rows)


df_csv_summary = build_keypoint_csv_summary(KEYPOINT_CSV_DIR)

df_csv_summary.to_csv(KEYPOINT_SUMMARY_PATH, index=False, encoding="utf-8-sig")

print("\nKeypoint CSV summary saved:")
print(KEYPOINT_SUMMARY_PATH)
print("Generated CSV count:", len(df_csv_summary))

display(df_csv_summary.head(10))


# ============================================================
# 6. Check Missing Keypoint CSV Files
# ============================================================

def check_missing_keypoints(df_clean, keypoint_csv_dir, skip_files=None):
    """
    Compare clean manifest videos against generated keypoint CSV files.
    """
    if skip_files is None:
        skip_files = set()

    expected = []
    missing = []

    for _, row in df_clean.iterrows():
        video_path = row["full_path"]
        video_name = os.path.splitext(os.path.basename(video_path))[0]

        if video_name in skip_files:
            continue

        expected_csv = os.path.join(keypoint_csv_dir, f"{video_name}_keypoints.csv")

        expected.append({
            "video_name": video_name,
            "expected_csv_path": expected_csv,
            "video_path": video_path,
        })

        if not os.path.exists(expected_csv):
            missing.append({
                "video_name": video_name,
                "missing_csv_path": expected_csv,
                "video_path": video_path,
            })

    return pd.DataFrame(expected), pd.DataFrame(missing)


df_expected, df_missing = check_missing_keypoints(
    df_clean=df_clean,
    keypoint_csv_dir=KEYPOINT_CSV_DIR,
    skip_files=SKIP_FILES,
)

df_missing.to_csv(MISSING_KEYPOINTS_PATH, index=False, encoding="utf-8-sig")

print("\nMissing keypoints report saved:")
print(MISSING_KEYPOINTS_PATH)
print("Expected CSV:", len(df_expected))
print("Missing CSV:", len(df_missing))

display(df_missing.head(10))


# ============================================================
# 7. Merge Keypoint Summary with Clean Manifest
# ============================================================

def merge_keypoint_summary_with_manifest(df_csv_summary, df_clean):
    """
    Merge generated keypoint CSV summary with original video metadata.
    """
    df_manifest_for_merge = df_clean.copy()

    df_manifest_for_merge["video_name"] = df_manifest_for_merge["full_path"].apply(
        lambda p: os.path.splitext(os.path.basename(p))[0]
    )

    df_merged = df_csv_summary.merge(
        df_manifest_for_merge,
        on="video_name",
        how="left",
    )

    return df_merged


df_merged = merge_keypoint_summary_with_manifest(df_csv_summary, df_clean)

df_merged.to_csv(MERGED_MANIFEST_PATH, index=False, encoding="utf-8-sig")

print("\nMerged manifest saved:")
print(MERGED_MANIFEST_PATH)
print("Merged shape:", df_merged.shape)
print("Columns:", df_merged.columns.tolist())

display(df_merged.head(10))


# ============================================================
# 8. Rule-Based Motion Feature Extraction and Classification
# ============================================================

def safe_col(df, col):
    """
    Return column if exists; otherwise return NaN series.
    """
    if col in df.columns:
        return df[col]
    return pd.Series([np.nan] * len(df))


def extract_motion_features(csv_path):
    """
    Extract simple motion features from a keypoint CSV.
    These are used only for rule-based pseudo classification
    and manual-labeling candidate selection.
    """
    df = pd.read_csv(csv_path)

    right_foot_y = safe_col(df, "right_foot_index_y")
    left_foot_y = safe_col(df, "left_foot_index_y")
    right_foot_x = safe_col(df, "right_foot_index_x")
    left_foot_x = safe_col(df, "left_foot_index_x")

    right_wrist_x = safe_col(df, "right_wrist_x")
    left_wrist_x = safe_col(df, "left_wrist_x")

    right_hip_y = safe_col(df, "right_hip_y")
    left_hip_y = safe_col(df, "left_hip_y")
    right_shoulder_y = safe_col(df, "right_shoulder_y")
    left_shoulder_y = safe_col(df, "left_shoulder_y")

    features = {}

    # Smaller y means higher position in image
    features["right_foot_highest"] = right_foot_y.min()
    features["left_foot_highest"] = left_foot_y.min()
    features["min_foot_y"] = min(features["right_foot_highest"], features["left_foot_highest"])

    # Foot movement range
    features["right_foot_x_range"] = right_foot_x.max() - right_foot_x.min()
    features["left_foot_x_range"] = left_foot_x.max() - left_foot_x.min()
    features["right_foot_y_range"] = right_foot_y.max() - right_foot_y.min()
    features["left_foot_y_range"] = left_foot_y.max() - left_foot_y.min()

    # Foot speed
    features["right_foot_speed"] = np.sqrt(
        right_foot_x.diff() ** 2 + right_foot_y.diff() ** 2
    ).mean()

    features["left_foot_speed"] = np.sqrt(
        left_foot_x.diff() ** 2 + left_foot_y.diff() ** 2
    ).mean()

    features["max_foot_speed"] = max(features["right_foot_speed"], features["left_foot_speed"])

    # Body vertical relation
    shoulder_center_y = (right_shoulder_y + left_shoulder_y) / 2
    hip_center_y = (right_hip_y + left_hip_y) / 2
    features["body_vertical_range"] = (hip_center_y - shoulder_center_y).mean()

    # Punch-like wrist movement
    features["right_wrist_x_range"] = right_wrist_x.max() - right_wrist_x.min()
    features["left_wrist_x_range"] = left_wrist_x.max() - left_wrist_x.min()
    features["max_wrist_x_range"] = max(
        features["right_wrist_x_range"],
        features["left_wrist_x_range"]
    )

    return features


def rule_based_classify(feat):
    """
    Simple rule-based classifier.
    This is NOT the final AI model.
    It is only used to create candidate groups for manual labeling.
    """
    min_foot_y = feat["min_foot_y"]
    foot_y_range = max(feat["right_foot_y_range"], feat["left_foot_y_range"])
    foot_x_range = max(feat["right_foot_x_range"], feat["left_foot_x_range"])
    foot_speed = feat["max_foot_speed"]
    wrist_range = feat["max_wrist_x_range"]

    # 1. Punch: clear hand movement, small foot movement
    if wrist_range > 0.22 and foot_y_range < 0.22:
        return "Punch / Hand Motion"

    # 2. Axe / high kick: very high foot and large vertical motion
    if min_foot_y < 0.35 and foot_y_range > 0.30:
        return "Axe Kick / High Kick"

    # 3. Side kick: strong horizontal leg extension, smaller vertical swing
    if foot_x_range > 0.28 and foot_y_range < 0.38 and foot_speed > 0.012:
        return "Side Kick"

    # 4. Roundhouse: both horizontal and vertical movement visible
    if foot_x_range > 0.30 and foot_y_range > 0.25 and foot_speed > 0.015:
        return "Roundhouse Kick"

    # 5. Front kick: mainly vertical leg lifting motion
    if foot_y_range > 0.20:
        return "Front Kick"

    return "Unknown"


def build_rule_based_results(df_csv_summary):
    """
    Build rule-based classification result table for all generated CSVs.
    """
    rows = []

    for i, row in df_csv_summary.iterrows():
        csv_path = row["csv_path"]

        try:
            feat = extract_motion_features(csv_path)
            pred = rule_based_classify(feat)

            rows.append({
                "video_name": row["video_name"],
                "csv_path": csv_path,
                "predicted_action": pred,
                **feat,
            })

        except Exception as e:
            rows.append({
                "video_name": row.get("video_name", ""),
                "csv_path": csv_path,
                "predicted_action": "ERROR",
                "error": str(e),
            })

    return pd.DataFrame(rows)


df_rule_results = build_rule_based_results(df_csv_summary)

df_rule_results.to_csv(RULE_BASED_RESULTS_PATH, index=False, encoding="utf-8-sig")

print("\nRule-based classification results saved:")
print(RULE_BASED_RESULTS_PATH)
print("Shape:", df_rule_results.shape)

print("\nRule-based class distribution:")
print(df_rule_results["predicted_action"].value_counts(dropna=False))

display(df_rule_results.head(10))


# ============================================================
# 9. Create Manual Labeling Template
# ============================================================

def create_manual_labeling_template(df_rule_results):
    """
    Create full manual labeling template from rule-based results.
    """
    keep_cols = [
        "video_name",
        "csv_path",
        "predicted_action",
        "max_foot_speed",
        "right_foot_y_range",
        "left_foot_y_range",
        "right_foot_x_range",
        "left_foot_x_range",
        "body_vertical_range",
    ]

    keep_cols = [c for c in keep_cols if c in df_rule_results.columns]

    df_label = df_rule_results[keep_cols].copy()

    df_label["manual_label"] = ""
    df_label["label_status"] = "unlabeled"
    df_label["notes"] = ""

    df_label = df_label.sort_values(
        by=["predicted_action", "video_name"],
        na_position="last"
    )

    return df_label


df_manual_template = create_manual_labeling_template(df_rule_results)

df_manual_template.to_csv(MANUAL_LABEL_TEMPLATE_PATH, index=False, encoding="utf-8-sig")

print("\nManual labeling template saved:")
print(MANUAL_LABEL_TEMPLATE_PATH)
print("Rows:", len(df_manual_template))

display(df_manual_template.head(10))


# ============================================================
# 10. Match Original Video Paths for Manual Labeling
# ============================================================

def collect_video_files(video_roots):
    """
    Collect original video files from possible video roots.
    """
    video_files = []

    for root in video_roots:
        if os.path.exists(root):
            for ext in VIDEO_EXTS:
                video_files.extend(
                    glob.glob(os.path.join(root, "**", f"*{ext}"), recursive=True)
                )

    return sorted(set(video_files))


def build_video_map(video_files):
    """
    Build mapping from filename stem to full video path.
    """
    video_map = {}

    for p in video_files:
        stem = os.path.splitext(os.path.basename(p))[0]
        video_map[stem] = p

    return video_map


def find_video_path(video_name, video_map):
    """
    Match keypoint CSV video_name back to original video path.
    """
    video_name = str(video_name)

    # Direct match
    if video_name in video_map:
        return video_map[video_name]

    # Remove _keypoints just in case
    clean_name = video_name.replace("_keypoints", "")

    if clean_name in video_map:
        return video_map[clean_name]

    # Fuzzy fallback
    for stem, path in video_map.items():
        if clean_name == stem:
            return path
        if clean_name in stem or stem in clean_name:
            return path

    return ""


VIDEO_ROOTS_FOR_MATCHING = [
    f"{ROOT}/taekwondo_videos",
    f"{ROOT}/original_data",
    f"{ROOT}/RBDA_Group_Project",
    ROOT,
]

video_files = collect_video_files(VIDEO_ROOTS_FOR_MATCHING)
video_map = build_video_map(video_files)

print("\nOriginal video files found for matching:", len(video_files))


df_manual_template_with_path = df_manual_template.copy()
df_manual_template_with_path["original_video_path"] = df_manual_template_with_path["video_name"].apply(
    lambda name: find_video_path(name, video_map)
)

matched_count = (df_manual_template_with_path["original_video_path"] != "").sum()
unmatched_count = (df_manual_template_with_path["original_video_path"] == "").sum()

print("Matched original videos:", matched_count)
print("Unmatched original videos:", unmatched_count)

print("\nMatched examples:")
display(df_manual_template_with_path[df_manual_template_with_path["original_video_path"] != ""].head(10))

print("\nUnmatched examples:")
display(df_manual_template_with_path[df_manual_template_with_path["original_video_path"] == ""].head(10))


# ============================================================
# 11. Create Manual Labeling Sample
# ============================================================

def create_manual_labeling_sample(df_template_with_path, output_path):
    """
    Create a balanced-ish sample for manual labeling.
    Uses rule-based predicted_action only to select candidate videos.
    Human label will be assigned later in 03_manual_labeling_colab.
    """
    target_actions = [
        "Front Kick",
        "Roundhouse Kick",
        "Side Kick",
        "Axe Kick / High Kick",
    ]

    samples = []

    for action in target_actions:
        part = df_template_with_path[
            (df_template_with_path["predicted_action"] == action) &
            (df_template_with_path["original_video_path"] != "")
        ].copy()

        n = min(30, len(part))

        print(f"{action}: available={len(part)}, sampled={n}")

        if n > 0:
            samples.append(part.sample(n=n, random_state=RANDOM_STATE))

    if not samples:
        raise RuntimeError("No samples created. Check rule-based results and video path matching.")

    df_sample = pd.concat(samples, ignore_index=True)

    # Reset manual labeling columns
    df_sample["manual_label"] = ""
    df_sample["label_status"] = "unlabeled"
    df_sample["notes"] = ""

    front_cols = [
        "video_name",
        "original_video_path",
        "csv_path",
        "predicted_action",
        "manual_label",
        "label_status",
        "notes",
    ]

    other_cols = [c for c in df_sample.columns if c not in front_cols]
    df_sample = df_sample[front_cols + other_cols]

    df_sample.to_csv(output_path, index=False, encoding="utf-8-sig")

    return df_sample


df_manual_sample = create_manual_labeling_sample(
    df_template_with_path=df_manual_template_with_path,
    output_path=MANUAL_LABEL_SAMPLE_PATH,
)

print("\nManual labeling sample saved:")
print(MANUAL_LABEL_SAMPLE_PATH)

print("\nSample size:", len(df_manual_sample))
print("\nSample rule-based distribution:")
print(df_manual_sample["predicted_action"].value_counts(dropna=False))

display(df_manual_sample.head(20))


# ============================================================
# 12. Final Output Summary
# ============================================================

print("\n============================================================")
print("01_data_processing_colab DONE")
print("Generated / updated files:")
print("============================================================")
print("1.", MANIFEST_PATH)
print("2.", CLEAN_MANIFEST_PATH)
print("3.", KEYPOINT_SUMMARY_PATH)
print("4.", MISSING_KEYPOINTS_PATH)
print("5.", MERGED_MANIFEST_PATH)
print("6.", RULE_BASED_RESULTS_PATH)
print("7.", MANUAL_LABEL_TEMPLATE_PATH)
print("8.", MANUAL_LABEL_SAMPLE_PATH)
print("============================================================")

print("\nRecommended next notebook:")
print("02_keypoint_extraction_colab if keypoints are missing")
print("03_manual_labeling_colab if keypoints and manual labeling sample are ready")